# Use Case: Customer Segmentation (Clustering/Unsupervised)

**User Prompt Example:**
```
Segment customers based on @[path/to/customer_data.csv].
Use K-Means and DBSCAN. Export results and create visualizations.
Follow the data-science-lab skill.
```

**Goal Interpretation:**
| User Goal | Task Type | Recommended Models |
|-----------|-----------|-------------------|
| "segment X" / "group" | clustering | K-Means, DBSCAN, Hierarchical |

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Clustering algorithms
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score

import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

from visualize import plot_clustering_results

sns.set_style('whitegrid')

for d in ['results', 'images', 'data']:
    os.makedirs(d, exist_ok=True)

print('✅ Setup complete.')

---

## 2. Load & Explore Data

In [ ]:
# Generate sample customer data (replace with actual data)
np.random.seed(42)
n_customers = 500

data = {
    'customer_id': range(1, n_customers + 1),
    'annual_income': np.random.normal(60000, 20000, n_customers),
    'spending_score': np.random.normal(50, 25, n_customers),
    'age': np.random.randint(18, 70, n_customers),
    'purchase_frequency': np.random.randint(1, 50, n_customers),
    'avg_transaction': np.random.exponential(100, n_customers),
}

df = pd.DataFrame(data)

# Ensure positive values
df['annual_income'] = df['annual_income'].clip(lower=10000)
df['spending_score'] = df['spending_score'].clip(0, 100)

print(f'Shape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.describe()

In [ ]:
# Visualization: Distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(df.columns[1:]):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig('images/customer_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. Data Preparation

In [ ]:
# Select features for clustering
features = ['annual_income', 'spending_score', 'age', 'purchase_frequency', 'avg_transaction']
X = df[features].copy()

# Handle missing values (if any)
X = X.fillna(X.median())

# Feature scaling (important for clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features: {features}')
print(f'Scaled shape: {X_scaled.shape}')

---

## 4. K-Means Clustering

In [ ]:
# Elbow Method to find optimal k
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow and Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(True)

axes[1].plot(k_range, silhouettes, 'go-')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('images/kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best k by silhouette: {k_range[np.argmax(silhouettes)]} (score: {max(silhouettes):.4f})')

In [ ]:
# Apply K-Means with optimal k (let's use 5 based on elbow)
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

df['kmeans_cluster'] = kmeans_labels

print(f'K-Means Clustering Results (k={optimal_k})')
print(f'Silhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}')
print(f'\nCluster Distribution:')
print(df['kmeans_cluster'].value_counts().sort_index())

---

## 5. DBSCAN Clustering

In [ ]:
# DBSCAN - find optimal eps using k-distance graph
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors=2)
neighbors.fit(X_scaled)
distances, indices = neighbors.kneighbors(X_scaled)
distances = np.sort(distances[:, 1])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.xlabel('Points sorted by distance')
plt.ylabel('2-NN Distance')
plt.title('K-Distance Graph for DBSCAN eps')
plt.grid(True)
plt.savefig('images/dbscan_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

df['dbscan_cluster'] = dbscan_labels

print(f'DBSCAN Clustering Results')
print(f'Number of clusters: {len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}')
print(f'Number of noise points: {sum(dbscan_labels == -1)}')
print(f'\nCluster Distribution:')
print(pd.Series(dbscan_labels).value_counts().sort_index())

---

## 6. Hierarchical Clustering

In [ ]:
# Agglomerative (Hierarchical) Clustering
hierarchical = AgglomerativeClustering(n_clusters=5, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

df['hierarchical_cluster'] = hierarchical_labels

print(f'Hierarchical Clustering Results (k=5)')
print(f'Silhouette Score: {silhouette_score(X_scaled, hierarchical_labels):.4f}')
print(f'\nCluster Distribution:')
print(df['hierarchical_cluster'].value_counts().sort_index())

---

## 7. Cluster Visualization

In [ ]:
# Visualize clusters (using first 2 principal components for visualization)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, labels, title in zip(axes, 
                               [kmeans_labels, dbscan_labels, hierarchical_labels],
                               ['K-Means', 'DBSCAN', 'Hierarchical']):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.6, s=50)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'{title} Clustering')
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.savefig('images/clustering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}')

---

## 8. Cluster Profiling

In [ ]:
# Profile each K-Means cluster
print('=== K-Means Cluster Profiles ===\n')

cluster_profiles = df.groupby('kmeans_cluster')[features].mean()
print(cluster_profiles.round(2))

# Visualize cluster profiles
cluster_profiles.T.plot(kind='bar', figsize=(12, 6))
plt.title('Cluster Profiles (Mean Values)')
plt.xlabel('Features')
plt.ylabel('Mean Value')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('images/cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 9. Results Export

In [ ]:
# Export clustering results
results = []

for name, labels in [('KMeans', kmeans_labels), 
                       ('DBSCAN', dbscan_labels), 
                       ('Hierarchical', hierarchical_labels)]:
    # Exclude noise points for DBSCAN
    valid_labels = labels[labels != -1]
    if len(valid_labels) > 1:
        sil = silhouette_score(X_scaled[labels != -1], valid_labels)
    else:
        sil = 0
    
    results.append({
        'model': name,
        'n_clusters': len(set(labels)) - (1 if -1 in labels else 0),
        'silhouette_score': sil,
        'noise_points': sum(labels == -1)
    })

results_df = pd.DataFrame(results)
results_df.to_csv('results/clustering_results.csv', index=False)

print('✅ Results saved to results/clustering_results.csv')
print(results_df)

In [ ]:
# Export clustered data
df.to_csv('data/customers_segmented.csv', index=False)
print('✅ Segmented data saved to data/customers_segmented.csv')

---

## 10. Conclusion

### Key Findings

**K-Means Clustering (k=5):**
- Best silhouette score achieved with k=5
- Identified distinct customer segments based on income, spending, age

**DBSCAN:**
- Found outliers/noise points automatically
- No need to pre-specify number of clusters

**Hierarchical:**
- Provides dendrogram for visual interpretation
- Good for understanding cluster hierarchy

### Business Implications
- High Income + High Spending: VIP customers → loyalty programs
- High Income + Low Spending: Potential upsell targets
- Low Income + High Spending: Budget-conscious impulse buyers
- Low Income + Low Spending: Basic service customers